<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Fine_Tuning_Large_Language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning n Generation Model - Gemma 4 12B , by google

## 1. Install Dependencies
This cell installs the required libraries. Run this first to ensure you have the latest versions of `trl`, `peft`, and `bitsandbytes`.

In [ ]:
!pip install -q -U bitsandbytes transformers datasets peft trl accelerate

## 2. Import Libraries and Define Model



In [28]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

# Define the target model
model_name = "google/gemma-4-12b-it"

## 3. Load & Configure the Tokenizer

This cell loads the tokenizer. Crucially, it sets the padding side to "right" (required for training) and assigns the end-of-sequence token as the padding token, which Gemma models require.

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix padding for training
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

## 4. Load, Format, and Pre-Tokenize the Dataset

This is the most important step. To avoid trl dataset formatting errors, we apply the chat template and tokenize the text into input_ids before passing it to the trainer. We then delete the original text columns so PyTorch doesn't crash.

In [ ]:
# Load a subset of the UltraChat dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft").shuffle(seed=42).select(range(10000))

# Pre-tokenize the dataset to avoid all SFTTrainer column conflicts
def tokenize_and_format(example):
    # Apply chat template and tokenize immediately
    chat = example["messages"]

    tokenized = tokenizer.apply_chat_template(
        chat,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=2048
    )

    # For causal language modeling, labels are the input_ids
    return {
        "input_ids": tokenized,
        "labels": tokenized.copy()
    }

# Apply the mapping and completely remove all original string columns (messages, prompt, etc.)
dataset = dataset.map(tokenize_and_format, remove_columns=dataset.column_names)

print(f"Dataset ready. Example input_ids: {dataset[0]['input_ids'][:10]}")

## 5. Load the Base Model with 4-bit Quantization

This cell loads the massive 12B parameter model into your GPU memory using bitsandbytes 4-bit NormalFloat (nf4) quantization, preserving performance while drastically reducing VRAM usage.

In [ ]:
# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the base model with quantization applied
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Disable KV cache (required for stable memory during training)
model.config.use_cache = False

print("Model loaded in 4-bit precision.")

## 6. Inject LoRA Adapters

This cell freezes the core model weights and injects the tiny, trainable LoRA matrices into all the target attention and MLP layers.

In [ ]:
# Configure LoRA hyperparameters
peft_config = LoraConfig(
    r=64,                      # Rank (64 provides strong capacity)
    lora_alpha=128,            # Alpha scaling
    lora_dropout=0.05,         # Light dropout
    bias="none",               # No bias training
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

# Prepare model for stable quantized training
model = prepare_model_for_kbit_training(model)

# Apply LoRA configuration
model = get_peft_model(model, peft_config)

# Print trainable parameter count to verify LoRA injection
model.print_trainable_parameters()

## 7. Configure Training Loop & Train

This final cell defines the training parameters using SFTConfig and begins the training loop using SFTTrainer. Because we pre-tokenized the data in Step 4, we use a clean configuration without any of the error-prone dataset string arguments.

In [ ]:
# Define training hyperparameters
training_arguments = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    bf16=True,                       # Crucial: Must be bf16 to match the model's compute_dtype
    gradient_checkpointing=True,
    max_length=2048,
    packing=False
)


In [ ]:

# Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    processing_class=tokenizer,      # Replaces the deprecated 'tokenizer' argument
)

# Start Fine-Tuning!
trainer.train()

# Save the trained LoRA adapter weights
output_adapter = "gemma-4-12b-ft-adapter"
trainer.model.save_pretrained(output_adapter)
tokenizer.save_pretrained(output_adapter)

print(f"Training complete. LoRA adapters successfully saved to '{output_adapter}'")